# Positional encoding
Self-attention ignores word order; the first ideas for adding positions and their problems; the sine-cosine
positional encoding of "Attention Is All You Need"; its heatmap; and why it carries relative positions.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import pandas as pd
import keras

np.set_printoptions(precision=4, suppress=True)


def positional_encoding(n_pos, d_model):
    """PE[pos, 2i] = sin(pos / 10000^(2i/d)), PE[pos, 2i+1] = cos(same angle) (Vaswani et al. 2017, 3.5)."""
    pos = np.arange(n_pos)[:, None]
    i = np.arange(d_model // 2)[None, :]
    angle = pos / 10000 ** (2 * i / d_model)
    pe = np.zeros((n_pos, d_model))
    pe[:, 0::2] = np.sin(angle)
    pe[:, 1::2] = np.cos(angle)
    return pe

## 1. Self-attention alone is blind to word order
One attention layer (random weights, one head) reads "rahul killed the lion" and "the lion killed rahul".
Each word has the same embedding in both sentences. Only the order differs.

In [2]:
keras.utils.set_random_seed(0)
words = ["rahul", "killed", "the", "lion"]
emb = {w: np.random.normal(size=6).astype("float32") for w in words}
attn = keras.layers.MultiHeadAttention(num_heads=1, key_dim=6)


def contextual(sentence, add_positions=False):
    x = np.stack([emb[w] for w in sentence])
    if add_positions:
        x = x + positional_encoding(len(sentence), 6).astype("float32")
    x = x[None]                                   # a batch of one sentence
    return dict(zip(sentence, attn(x, x).numpy()[0]))


s1 = ["rahul", "killed", "the", "lion"]
s2 = ["the", "lion", "killed", "rahul"]
a, b = contextual(s1), contextual(s2)
for w in words:
    print(f"{w:7s} sentence 1: {a[w]}   sentence 2: {b[w]}")
assert all(np.allclose(a[w], b[w], atol=1e-6) for w in words)
print("Same contextual vector for every word in both orders.")

rahul   sentence 1: [ 2.3955 -1.0815  3.0366  0.5248  0.8863 -1.3707]   sentence 2: [ 2.3955 -1.0815  3.0366  0.5248  0.8863 -1.3707]
killed  sentence 1: [ 1.176  -0.0372  1.5686  1.2613  0.412  -1.2655]   sentence 2: [ 1.176  -0.0372  1.5686  1.2613  0.412  -1.2655]
the     sentence 1: [ 1.9344 -0.8029  2.4245  0.7032  0.6815 -1.1816]   sentence 2: [ 1.9344 -0.8029  2.4245  0.7032  0.6815 -1.1816]
lion    sentence 1: [ 1.1421  0.4253  1.7071  1.7019  0.4858 -1.8174]   sentence 2: [ 1.1421  0.4253  1.7071  1.7019  0.4858 -1.8174]
Same contextual vector for every word in both orders.


The same check with a random shuffle of a longer real sentence: the outputs are the input rows shuffled the same way.

In [3]:
rng = np.random.default_rng(1)
x = rng.normal(size=(1, 12, 6)).astype("float32")          # 12 words
perm = rng.permutation(12)
out = attn(x, x).numpy()[0]
out_shuffled = attn(x[:, perm], x[:, perm]).numpy()[0]
print("max |difference| after undoing the shuffle:", np.abs(out_shuffled - out[perm]).max())

max |difference| after undoing the shuffle: 1.7881393e-07


With positional encodings added, the two sentences give different vectors for the same word.

In [4]:
a, b = contextual(s1, True), contextual(s2, True)
for w in words:
    print(f"{w:7s} change between the two orders: {np.abs(a[w] - b[w]).max():.3f}")

rahul   change between the two orders: 1.321
killed  change between the two orders: 0.538
the     change between the two orders: 1.679
lion    change between the two orders: 0.333


## 2. First ideas: counting, and counting divided by the length

In [5]:
book_words = 500 * 200                       # 500 pages of 200 words
print("last position in a 500-page book:", book_words)
for sent in (["thank", "you"], ["rahul", "killed", "the", "lion"]):
    n = len(sent)
    print(sent, "positions / length:", [round(p / n, 2) for p in range(1, n + 1)])

last position in a 500-page book: 100000
['thank', 'you'] positions / length: [0.5, 1.0]
['rahul', 'killed', 'the', 'lion'] positions / length: [0.25, 0.5, 0.75, 1.0]


## 3. One sine wave: bounded and smooth, but it repeats

In [6]:
print("sin(1..4):", np.sin(np.arange(1, 5)).round(2))
print("cos(1..4):", np.cos(np.arange(1, 5)).round(2))


def closest_pair(enc):
    """Smallest Euclidean distance between the encodings of two different positions."""
    d = np.sqrt(((enc[:, None, :] - enc[None, :, :]) ** 2).sum(-1))
    d[np.diag_indices(len(enc))] = np.inf
    p, q = np.unravel_index(d.argmin(), d.shape)
    return d.min(), min(p, q), max(p, q)


pos = np.arange(1, 1001)[:, None].astype(float)
variants = {
    "sin(pos)": np.sin(pos),
    "sin, cos of pos": np.hstack([np.sin(pos), np.cos(pos)]),
    "+ sin, cos of pos/2": np.hstack([np.sin(pos), np.cos(pos), np.sin(pos / 2), np.cos(pos / 2)]),
    "rates 1, 1/10, 1/100": np.hstack([f(pos / v) for v in (1, 10, 100) for f in (np.sin, np.cos)]),
}
rows = []
for name, enc in variants.items():
    dist, p, q = closest_pair(enc)
    rows.append((name, enc.shape[1], p + 1, q + 1, dist))
    print(f"{name:22s} {enc.shape[1]} numbers: closest positions {p + 1} and {q + 1}, distance {dist:.2g}")
pd.DataFrame(rows, columns=["encoding", "numbers", "pos_a", "pos_b", "distance"]).to_csv("data/closest_pairs.csv", index=False)
print("sin(11) =", np.sin(11), " sin(344) =", np.sin(344))
print("775 - 398 =", 775 - 398, "  60 * 2 pi =", round(60 * 2 * np.pi, 2))

sin(1..4): [ 0.84  0.91  0.14 -0.76]
cos(1..4): [ 0.54 -0.42 -0.99 -0.65]
sin(pos)               1 numbers: closest positions 11 and 344, distance 1.3e-07
sin, cos of pos        2 numbers: closest positions 15 and 725, distance 6e-05
+ sin, cos of pos/2    4 numbers: closest positions 398 and 775, distance 0.0099
rates 1, 1/10, 1/100   6 numbers: closest positions 51 and 679, distance 0.32
sin(11) = -0.9999902065507035  sin(344) = -0.9999903395061709
775 - 398 = 377   60 * 2 pi = 376.99


## 4. The formula of the paper, on "river bank" with d_model = 6

In [7]:
pe6 = positional_encoding(2, 6)
print("river (pos 0):", pe6[0].round(4))
print("bank  (pos 1):", pe6[1].round(4))
print("angle rates 1/10000^(2i/6):", (1 / 10000 ** (2 * np.arange(3) / 6)).round(5))

river (pos 0): [0. 1. 0. 1. 0. 1.]
bank  (pos 1): [0.8415 0.5403 0.0464 0.9989 0.0022 1.    ]
angle rates 1/10000^(2i/6): [1.     0.0464 0.0022]


Wavelengths of the 256 sine-cosine pairs when d_model = 512: from 2*pi to almost 10000*2*pi.

In [8]:
wl = 2 * np.pi * 10000 ** (2 * np.arange(256) / 512)
print("shortest", wl[0].round(2), " longest", wl[-1].round(0), " 10000*2pi =", round(10000 * 2 * np.pi))
print("ratio between neighbours:", (wl[1] / wl[0]).round(4), "(constant: a geometric progression)")

shortest 6.28  longest 60611.0  10000*2pi = 62832
ratio between neighbours: 1.0366 (constant: a geometric progression)


## 5. The heatmap: 50 positions, d_model = 128

In [9]:
pe = positional_encoding(50, 128)
print("first row (position 0), first 8 values:", pe[0, :8])
spread = pe.max(0) - pe.min(0)
print("range over the 50 positions, dimensions 0-7:  ", spread[:8].round(2))
print("range over the 50 positions, dimensions 120-127:", spread[120:].round(4))

first row (position 0), first 8 values: [0. 1. 0. 1. 0. 1. 0. 1.]
range over the 50 positions, dimensions 0-7:   [2. 2. 2. 2. 2. 2. 2. 2.]
range over the 50 positions, dimensions 120-127: [0.0087 0.     0.0075 0.     0.0065 0.     0.0057 0.    ]


## 6. Relative position: one matrix per offset k
For each sine-cosine pair with angle rate w, the 2x2 rotation [[cos wk, sin wk], [-sin wk, cos wk]] takes the pair at
position p to the pair at position p + k, whatever p is. M_k stacks these blocks along the diagonal.

In [10]:
d = 128
rates = 1 / 10000 ** (2 * np.arange(d // 2) / d)
PE = positional_encoding(200, d)


def M(k):
    m = np.zeros((d, d))
    for i, w in enumerate(rates):
        c, s = np.cos(w * k), np.sin(w * k)
        m[2 * i:2 * i + 2, 2 * i:2 * i + 2] = [[c, s], [-s, c]]
    return m


for k, starts in ((10, (10, 30, 40)), (5, (5, 12, 21))):
    for p in starts:
        err = np.abs(M(k) @ PE[p] - PE[p + k]).max()
        print(f"M_{k} @ PE({p}) vs PE({p + k}): max |difference| = {err:.1e}")

M_10 @ PE(10) vs PE(20): max |difference| = 1.7e-15
M_10 @ PE(30) vs PE(40): max |difference| = 3.6e-15
M_10 @ PE(40) vs PE(50): max |difference| = 3.8e-15
M_5 @ PE(5) vs PE(10): max |difference| = 6.7e-16
M_5 @ PE(12) vs PE(17): max |difference| = 1.4e-15
M_5 @ PE(21) vs PE(26): max |difference| = 3.0e-15


The dot product PE(p) . PE(p+k) equals the sum of cos(w_i k): it depends on k only.

In [11]:
dots = PE @ PE.T
for k in (0, 1, 5, 10, 50):
    vals = np.array([dots[p, p + k] for p in range(200 - k)])
    formula = np.cos(rates * k).sum()
    print(f"k={k:2d}: dot over all p: min {vals.min():.4f}, max {vals.max():.4f}; sum of cos(w k) = {formula:.4f}")
kk = np.arange(0, 100)
pd.DataFrame({"k": kk, "dot": [np.cos(rates * k).sum() for k in kk]}).to_csv("data/dot_vs_offset.csv", index=False)

k= 0: dot over all p: min 64.0000, max 64.0000; sum of cos(w k) = 64.0000
k= 1: dot over all p: min 62.0937, max 62.0937; sum of cos(w k) = 62.0937
k= 5: dot over all p: min 47.1850, max 47.1850; sum of cos(w k) = 47.1850
k=10: dot over all p: min 42.8200, max 42.8200; sum of cos(w k) = 42.8200
k=50: dot over all p: min 34.9550, max 34.9550; sum of cos(w k) = 34.9550
